In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
DATA_DIR = "experiments"
FINAL_STEP = 156

In [ ]:
def load_behaviorspace_table(filename, columns):
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path, skiprows=6)
    df.columns = columns
    return df

In [ ]:
def welch_ttest_summary(group_a, group_b, label_a="A", label_b="B"):
    t_stat, p_val = stats.ttest_ind(group_a, group_b, equal_var=False)
    reduction_pct = (group_b.mean() - group_a.mean()) / group_b.mean() * 100
    return {
        f"{label_a}_mean": group_a.mean(),
        f"{label_a}_n": len(group_a),
        f"{label_b}_mean": group_b.mean(),
        f"{label_b}_n": len(group_b),
        "t_stat": t_stat,
        "p_value": p_val,
        "pct_change": reduction_pct,
    }

In [ ]:
e0 = load_behaviorspace_table(
    "E0-baseline-validation-table.csv",
    ["run", "fert", "fuel", "panic", "govt", "step", "bwr_f", "bwr_r", "fsi"],
)

e1 = load_behaviorspace_table(
    "E1-shock-decomposition-table.csv",
    ["run", "fert", "fuel", "panic", "govt", "step", "bwr_f", "bwr_r", "fsi", "demand"],
)

e2 = load_behaviorspace_table(
    "E2-intervention-returns-panic-table.csv",
    ["run", "govt", "rho", "panic", "fert", "fuel", "step", "bwr_f", "bwr_r", "fsi"],
)

fsi_exp = load_behaviorspace_table(
    "FSI-threshold-check-table.csv",
    ["run", "govt", "thresh", "rho", "panic", "fert", "fuel", "step",
     "bwr_f", "bwr_r", "fsi_idx"],
)

In [ ]:
e0_final = e0[e0["step"] == FINAL_STEP - 1]
print(e0_final.shape[0])
print(e0_final["bwr_f"].mean())
print(e0_final["bwr_f"].std())

In [ ]:
e1_final = e1[e1["step"] == FINAL_STEP]

for factor, on_val, off_val, label in [
    ("fert", 0.627, 1.0, "Fertilizer ban"),
    ("fuel", 0.75, 1.0, "Fuel shortage"),
    ("panic", 1.02, 1.0, "Consumer panic"),
]:
    on = e1_final[e1_final[factor] == on_val]["bwr_f"]
    off = e1_final[e1_final[factor] == off_val]["bwr_f"]
    t_stat, p_val = stats.ttest_ind(on, off, equal_var=False)
    print(label, on.mean(), len(on), off.mean(), len(off), p_val)

In [ ]:
e1_grouped = (
    e1_final.groupby(["fert", "fuel", "panic"])["bwr_f"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
e1_grouped

In [ ]:
e2_final = e2[e2["step"] == FINAL_STEP]
off_govt = e2_final[e2_final["govt"] == False]
rho_grouped = off_govt.groupby("rho")["bwr_f"].agg(["mean", "std", "count"])
rho_grouped

In [ ]:
r0 = off_govt[off_govt["rho"] == 0.0]["bwr_f"]
r1 = off_govt[off_govt["rho"] == 0.1]["bwr_f"]
r2 = off_govt[off_govt["rho"] == 0.2]["bwr_f"]
_, p01 = stats.ttest_ind(r0, r1, equal_var=False)
_, p02 = stats.ttest_ind(r0, r2, equal_var=False)
print(p01, p02)

In [ ]:
baseline = e2_final[(e2_final["rho"] == 0.1) & (e2_final["panic"] == 1.02)]
on = baseline[baseline["govt"] == True]["bwr_f"]
off = baseline[baseline["govt"] == False]["bwr_f"]
result = welch_ttest_summary(on, off, "ON", "OFF")
result

In [ ]:
fsi_final = fsi_exp[fsi_exp["step"] == FINAL_STEP]

threshold_results = {}
for t in [0.2, 0.4, 0.6]:
    on = fsi_final[(fsi_final["thresh"] == t) & (fsi_final["govt"] == True)]["bwr_f"]
    off = fsi_final[(fsi_final["thresh"] == t) & (fsi_final["govt"] == False)]["bwr_f"]
    result = welch_ttest_summary(on, off, "ON", "OFF")
    threshold_results[t] = result
    print(t, result["OFF_mean"], result["ON_mean"], result["pct_change"], result["p_value"])

In [ ]:
threshold_results